# 09 · Convolution and deconvolution / Convolución y deconvolución

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb)

*Part IV · exercise · 15 min*

This notebook follows one idea from a forward operation to an inverse problem:

> **A small kernel is reused across positions to transform an image.**

We will distinguish **correlation**, **convolution**, **transposed convolution**, and **deconvolution** without treating them as the same operation.

> 🇪🇸 Este cuaderno sigue una misma idea desde una operación directa hasta un problema inverso:
>
> **Un kernel pequeño se reutiliza en muchas posiciones para transformar una imagen.**
>
> Distinguiremos **correlación**, **convolución**, **convolución transpuesta** y **deconvolución** sin tratarlas como si fueran la misma operación.

## What you will be able to do / Lo que podrás hacer

- Explain what a **kernel / filtro** does using a real photograph.
- Predict `valid`, `same`, and `full` output shapes.
- Explain the difference between **cross-correlation** and true **convolution**.
- See a 1D convolution as ordinary matrix multiplication with a Toeplitz matrix.
- Explain transposed convolution as **overlap-add**, not as a magical inverse.
- Blur and partially recover a real image using **Richardson–Lucy deconvolution**.
- Explain why noise and image boundaries make inverse problems difficult.

> 🇪🇸
>
> - Explicar qué hace un **kernel / filtro** usando una fotografía real.
> - Predecir las formas de salida `valid`, `same` y `full`.
> - Explicar la diferencia entre **correlación cruzada** y **convolución** verdadera.
> - Ver una convolución 1D como multiplicación matricial con una matriz Toeplitz.
> - Explicar la convolución transpuesta como **superposición y suma**, no como una inversa mágica.
> - Desenfocar y recuperar parcialmente una imagen real con **deconvolución Richardson–Lucy**.
> - Explicar por qué el ruido y los bordes hacen difícil un problema inverso.

## Start with an everyday analogy / Empecemos con una analogía cotidiana

Imagine moving a small magnifying window across a photograph.

At every position:

1. the window covers a few nearby pixels;
2. each covered pixel is multiplied by a kernel weight;
3. the weighted values are added;
4. one output value is produced.

Then the same small rule moves to the next position.

That repeated local calculation is the main idea behind convolution-like image filtering.

### Four words to keep separate / Cuatro conceptos que debemos separar

| Term / Término | Simple idea / Idea sencilla |
|---|---|
| **Correlation / Correlación** | slide the kernel as written / deslizar el kernel tal como está |
| **Convolution / Convolución** | flip the kernel, then slide it / invertir el kernel y después deslizarlo |
| **Transposed convolution / Convolución transpuesta** | spread each input value through the kernel using overlap-add / distribuir cada entrada con el kernel mediante superposición y suma |
| **Deconvolution / Deconvolución** | try to recover an unknown original from a blurred observation / intentar recuperar un original desconocido a partir de una observación desenfocada |

> **Changing shape is not the same as recovering lost information.**

> 🇪🇸
>
> **Cambiar la forma no es lo mismo que recuperar información perdida.**

## `valid`, `same`, and `full` in plain language / `valid`, `same` y `full` en lenguaje sencillo

Suppose the image is `160×160` and the kernel is `3×3`.

### `valid`
Use only positions where the kernel fits completely inside the image.

Result:

`158×158`

### `same`
Return an output with the same spatial size as the input.

Result:

`160×160`

Values near the border depend on the boundary convention.

### `full`
Include every partial overlap between image and kernel.

Result:

`162×162`

> 🇪🇸
>
> - **`valid`**: el kernel debe caber completamente dentro de la imagen.
> - **`same`**: la salida conserva el mismo tamaño espacial de la entrada.
> - **`full`**: incluye también las superposiciones parciales de los bordes.

## Setup / Preparación

We use only real measured pixels from `skimage.data.camera()` for the image exercises.

From that real photograph we extract:

- a `160×160` crop for edge filtering;
- a `256×256` crop for deconvolution;
- a 32-pixel scanline for the Toeplitz view;
- a tiny `2×2` patch to make transposed-convolution overlap visible.

The Sobel and derivative kernels are **transformation rules**, not observed data.

> 🇪🇸 Usamos únicamente píxeles reales de `skimage.data.camera()` para los ejercicios de imagen.
>
> De esa fotografía extraemos un recorte `160×160`, otro `256×256`, una fila de 32 píxeles y un pequeño recorte `2×2`.
>
> Los kernels Sobel y de derivada son **reglas de transformación**, no datos observados.

In [ ]:
import numpy as np
from scipy import signal
from scipy.linalg import toeplitz
from skimage import data
from skimage.restoration import richardson_lucy
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

img = data.camera().astype(float) / 255.0
patch = img[176:336, 176:336]
work = img[128:384, 128:384]
scanline = img[256, 220:252].copy()
small = img[250:254:2, 250:254:2].copy()

sobel_x = np.array([
    [-1., 0., 1.],
    [-2., 0., 2.],
    [-1., 0., 1.],
])

kernel_1d = np.array([1., 0., -1.])

def convmtx_full_1d(kernel, n):
    m = len(kernel)

    col = np.zeros(n + m - 1)
    col[:m] = kernel

    row = np.zeros(n)
    row[0] = kernel[0]

    return toeplitz(col, row)

def transposed_overlap_add(x, kernel, stride=1):
    h, w = x.shape
    kh, kw = kernel.shape

    out_h = (h - 1) * stride + kh
    out_w = (w - 1) * stride + kw

    out = np.zeros(
        (out_h, out_w),
        dtype=float,
    )

    for i in range(h):
        for j in range(w):
            r = i * stride
            c = j * stride

            out[
                r:r + kh,
                c:c + kw,
            ] += x[i, j] * kernel

    return out

print("Full image / Imagen completa:", img.shape)
print("Filtering crop / Recorte para filtro:", patch.shape)
print("Deconvolution crop / Recorte de deconvolución:", work.shape)
print("Real scanline / Fila real:", scanline.shape)
print("Small real patch / Recorte real pequeño:", small.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## Why this matters / Por qué esto importa

Convolution is a **structured linear operator**.

The same local kernel is reused again and again.

That connects several ideas:

1. **Correlation ↔ convolution**  
   They differ by whether the kernel is flipped.

2. **Convolution ↔ matrix multiplication**  
   The repeated local rule can be written as a structured Toeplitz matrix.

3. **Forward blur ↔ inverse problem**  
   A blur can suppress information. Recovering that information under noise is much harder than applying the blur.

4. **Transposed convolution ↔ adjoint-style operation**  
   It spreads values back through the same local geometry, but it does not automatically reconstruct what was previously lost.

### Learning cycle / Ciclo de aprendizaje

Before each operation:

**Predict → Run → Explain / Predice → Ejecuta → Explica**

Ask:

- What is the input shape?
- What is the kernel shape?
- Will the output grow, shrink, or stay the same?
- Is information being filtered, rearranged, or estimated?

> 🇪🇸 La convolución es un **operador lineal estructurado**.
>
> La misma regla local se reutiliza muchas veces. Eso conecta correlación, multiplicación matricial, convolución transpuesta y problemas inversos.

### Interactive operation translator / Traductor interactivo de operaciones

Choose an operation and read what it really means.

> 🇪🇸 Elige una operación y observa qué significa realmente.

In [ ]:
operation_choice = widgets.Dropdown(
    options=[
        ("Correlation / Correlación", "correlation"),
        ("Convolution / Convolución", "convolution"),
        ("Transposed convolution / Convolución transpuesta", "transpose"),
        ("Deconvolution / Deconvolución", "deconvolution"),
    ],
    value="correlation",
    description="Operation / Operación:",
    style={"description_width": "145px"},
)

def explain_operation(operation):
    explanations = {
        "correlation": (
            "Slide the kernel as written.",
            "Desliza el kernel tal como está.",
            "forward filtering / filtrado directo",
        ),
        "convolution": (
            "Flip the kernel, then slide and sum.",
            "Invierte el kernel y después desliza y suma.",
            "forward filtering / filtrado directo",
        ),
        "transpose": (
            "Spread every input value through the kernel and add overlaps.",
            "Distribuye cada entrada mediante el kernel y suma las superposiciones.",
            "adjoint-style shape-changing operator / operador adjunto que cambia forma",
        ),
        "deconvolution": (
            "Estimate an unknown original from a blurred/noisy observation.",
            "Estima un original desconocido a partir de una observación desenfocada y ruidosa.",
            "inverse problem / problema inverso",
        ),
    }

    en, es, role = explanations[operation]

    print("EN:", en)
    print("ES:", es)
    print("Role / Papel:", role)

operation_output = widgets.interactive_output(
    explain_operation,
    {"operation": operation_choice},
)

display(widgets.VBox([operation_choice, operation_output]))

## Exercise 1 — correlation, convolution, and Toeplitz / Ejercicio 1 — correlación, convolución y Toeplitz

We start with real pixels.

For the `160×160` crop and `3×3` Sobel kernel:

- `valid` → `(158,158)`
- `same` → `(160,160)`
- `full` → `(162,162)`

For the real 32-pixel scanline and a length-3 kernel:

`full length = 32 + 3 - 1 = 34`

### Correlation vs convolution / Correlación vs convolución

To make true convolution produce the same output as correlation with `sobel_x`, use the **flipped** kernel:

`convolve2d(image, flip(sobel_x))`

### Toeplitz view / Vista Toeplitz

The 1D convolution can also be written as:

`C @ scanline`

where `C` is a structured matrix that repeats shifted copies of the same kernel.

> 🇪🇸 La convolución 1D de una fila real de píxeles también puede escribirse como `C @ scanline`.
>
> La matriz `C` no contiene una nueva operación: simplemente organiza las mismas multiplicaciones locales en forma matricial.

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Compute correlation with sobel_x using mode="valid".
# 2. Compute convolution using np.flip(sobel_x).
# 3. Verify that the results match.
# 4. Predict and verify valid, same, and full output shapes.
#
# ES:
# 1. Calcula correlación con sobel_x usando mode="valid".
# 2. Calcula convolución usando np.flip(sobel_x).
# 3. Verifica que los resultados coincidan.
# 4. Predice y verifica las formas valid, same y full.
#
# TODO 2 / TAREA 2
#
# EN:
# 1. Build C = convmtx_full_1d(kernel_1d, len(scanline)).
# 2. Compute C @ scanline.
# 3. Compare it with np.convolve(..., mode="full").
#
# ES:
# 1. Construye C = convmtx_full_1d(kernel_1d, len(scanline)).
# 2. Calcula C @ scanline.
# 3. Compáralo con np.convolve(..., mode="full").

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

corr = signal.correlate2d(
    patch,
    sobel_x,
    mode="valid",
)

conv_flipped = signal.convolve2d(
    patch,
    np.flip(sobel_x),
    mode="valid",
)

print(
    "Correlation == convolution with flipped kernel / "
    "Correlación == convolución con kernel invertido:",
    np.allclose(corr, conv_flipped),
)

print()

for mode in ["valid", "same", "full"]:
    result = signal.correlate2d(
        patch,
        sobel_x,
        mode=mode,
    )
    print(
        f"{mode:5s} -> shape/forma={result.shape}"
    )

C = convmtx_full_1d(
    kernel_1d,
    len(scanline),
)

via_matrix = C @ scanline

via_convolution = np.convolve(
    scanline,
    kernel_1d,
    mode="full",
)

print()
print("Toeplitz matrix / Matriz Toeplitz:", C.shape)
print("Full convolution / Convolución full:", via_convolution.shape)
print(
    "C @ scanline == convolution / C @ fila == convolución:",
    np.allclose(via_matrix, via_convolution),
)

print()
print("EN: the Toeplitz matrix is another representation of the same linear operation.")
print("ES: la matriz Toeplitz es otra representación de la misma operación lineal.")

### Interactive kernel-flip explorer / Explorador interactivo del volteo del kernel

Switch between:

- the Sobel kernel as written;
- the flipped Sobel kernel.

Then compare correlation and convolution outputs.

> 🇪🇸 Cambia entre el kernel Sobel original y el kernel invertido. Después compara las salidas de correlación y convolución.

In [ ]:
kernel_view = widgets.ToggleButtons(
    options=[
        ("Original Sobel", "original"),
        ("Flipped / Invertido", "flipped"),
    ],
    value="original",
    description="Kernel:",
    style={"description_width": "80px"},
)

def explore_kernel_flip(view):
    shown_kernel = (
        sobel_x
        if view == "original"
        else np.flip(sobel_x)
    )

    corr_live = signal.correlate2d(
        patch,
        sobel_x,
        mode="same",
    )

    conv_live = signal.convolve2d(
        patch,
        np.flip(sobel_x),
        mode="same",
    )

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(13, 3.4),
        constrained_layout=True,
    )

    axes[0].imshow(
        patch,
        cmap="gray",
    )
    axes[0].set_title("Real patch / Recorte real")

    im1 = axes[1].imshow(
        shown_kernel,
        cmap="coolwarm",
    )
    axes[1].set_title(
        "Kernel as shown / Kernel mostrado"
    )

    axes[2].imshow(
        corr_live,
        cmap="gray",
    )
    axes[2].set_title("Correlation / Correlación")

    axes[3].imshow(
        conv_live,
        cmap="gray",
    )
    axes[3].set_title(
        "Convolution with flipped kernel\n"
        "Convolución con kernel invertido"
    )

    axes[0].axis("off")
    axes[2].axis("off")
    axes[3].axis("off")

    for r in range(3):
        for c in range(3):
            axes[1].text(
                c,
                r,
                f"{shown_kernel[r, c]:.0f}",
                ha="center",
                va="center",
            )

    axes[1].set_xticks([])
    axes[1].set_yticks([])

    fig.colorbar(
        im1,
        ax=axes[1],
        fraction=0.046,
        pad=0.04,
    )

    plt.show()

    print(
        "Correlation == convolution with flipped kernel / "
        "Correlación == convolución con kernel invertido:",
        np.allclose(corr_live, conv_live),
    )
    print("EN: true convolution flips the kernel relative to cross-correlation.")
    print("ES: la convolución verdadera invierte el kernel respecto a la correlación cruzada.")

kernel_output = widgets.interactive_output(
    explore_kernel_flip,
    {"view": kernel_view},
)

display(widgets.VBox([kernel_view, kernel_output]))

### Interactive output-mode explorer / Explorador interactivo del modo de salida

Change `valid`, `same`, and `full`.

Watch both:

- the output image;
- the numerical output shape.

> 🇪🇸 Cambia entre `valid`, `same` y `full` y observa tanto la imagen como la forma numérica de salida.

In [ ]:
mode_widget = widgets.ToggleButtons(
    options=["valid", "same", "full"],
    value="same",
    description="Mode / Modo:",
    style={"description_width": "95px"},
)

def explore_mode(mode):
    filtered = signal.correlate2d(
        patch,
        sobel_x,
        mode=mode,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(8.2, 3.4),
        constrained_layout=True,
    )

    axes[0].imshow(
        patch,
        cmap="gray",
    )
    axes[0].set_title(
        f"Input / Entrada\n{patch.shape}"
    )

    axes[1].imshow(
        filtered,
        cmap="gray",
    )
    axes[1].set_title(
        f"Sobel correlation — {mode}\n"
        f"{filtered.shape}"
    )

    for ax in axes:
        ax.axis("off")

    plt.show()

    print("Input / Entrada:", patch.shape)
    print("Kernel:", sobel_x.shape)
    print("Output / Salida:", filtered.shape)

    explanations = {
        "valid": (
            "Only complete kernel overlaps are kept.",
            "Solo se conservan posiciones donde el kernel cabe completamente.",
        ),
        "same": (
            "The output keeps the input spatial size.",
            "La salida conserva el tamaño espacial de la entrada.",
        ),
        "full": (
            "Every partial boundary overlap is included.",
            "Se incluyen todas las superposiciones parciales de los bordes.",
        ),
    }

    en, es = explanations[mode]
    print("EN:", en)
    print("ES:", es)

mode_output = widgets.interactive_output(
    explore_mode,
    {"mode": mode_widget},
)

display(widgets.VBox([mode_widget, mode_output]))

### Interactive Toeplitz explorer / Explorador interactivo de Toeplitz

The 32 real pixel values form a vector `x`.

The convolution matrix has shape:

`C.shape = (34,32)`

Each row of `C` tells us which scanline values contribute to one output position.

Move **Output index / Índice de salida** and inspect one matrix row at a time.

> 🇪🇸 Los 32 valores reales forman un vector `x`.
>
> La matriz de convolución tiene forma `(34,32)`.
>
> Cada fila de `C` indica qué píxeles contribuyen a una posición de salida.

In [ ]:
toeplitz_index = widgets.IntSlider(
    value=8,
    min=0,
    max=len(via_matrix) - 1,
    step=1,
    description="Output index / Índice:",
    continuous_update=False,
    style={"description_width": "135px"},
)

def explore_toeplitz(k):
    row = C[k]
    contributions = row * scanline
    result = contributions.sum()

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(13, 3.5),
        constrained_layout=True,
    )

    axes[0].plot(
        np.arange(len(scanline)),
        scanline,
        marker="o",
    )
    axes[0].set_title("Real scanline / Fila real")
    axes[0].set_xlabel("pixel position / posición")

    axes[1].bar(
        np.arange(len(row)),
        row,
    )
    axes[1].set_title(
        f"Toeplitz row C[{k}] / Fila Toeplitz"
    )
    axes[1].set_xlabel("input position / posición entrada")

    axes[2].bar(
        np.arange(len(contributions)),
        contributions,
    )
    axes[2].set_title(
        "Products before summing / "
        "Productos antes de sumar"
    )
    axes[2].set_xlabel("input position / posición entrada")

    plt.show()

    print("Matrix result / Resultado matricial:", float(via_matrix[k]))
    print("Direct convolution / Convolución directa:", float(via_convolution[k]))
    print("Sum of shown contributions / Suma de contribuciones:", float(result))
    print()
    print("EN: one Toeplitz row is one shifted copy of the local convolution rule.")
    print("ES: una fila Toeplitz es una copia desplazada de la regla local de convolución.")

toeplitz_output = widgets.interactive_output(
    explore_toeplitz,
    {"k": toeplitz_index},
)

display(widgets.VBox([toeplitz_index, toeplitz_output]))

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

Three representations described closely related forward operations:

1. **Cross-correlation** slides the kernel as written.
2. **True convolution** flips the kernel first.
3. **Toeplitz multiplication** writes the repeated convolution rule as ordinary matrix multiplication.

Nothing magical happened when we introduced `C`.

We only changed the representation of the same linear operator.

> 🇪🇸 Correlación, convolución y multiplicación Toeplitz muestran distintas formas de representar una regla lineal repetida.

</details>

## Exercise 2 — transposed convolution changes shape, not history / Ejercicio 2 — la convolución transpuesta cambia forma, no historia

The phrase **“deconvolution layer”** is sometimes used informally for transposed convolution.

That name can be misleading.

A transposed-convolution-style operation does this:

1. take one input value;
2. multiply the kernel by that value;
3. place the scaled kernel in the output;
4. repeat for every input location;
5. add overlapping contributions.

This is **overlap-add**.

### Output shape / Forma de salida

For an input of height `H`, kernel size `K`, and stride `S`:

`output = (H - 1) × S + K`

For our real `2×2` input and `2×2` kernel:

- stride `1` → `3×3`
- stride `2` → `4×4`
- stride `3` → `5×5`

A larger output does **not** mean that previously lost image detail has been recovered.

> 🇪🇸 La convolución transpuesta distribuye cada valor de entrada a través del kernel y suma las zonas superpuestas.
>
> Puede aumentar el tamaño espacial, pero **aumentar el tamaño no equivale a recuperar información perdida**.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Inspect the real 2×2 patch `small`.
# 2. Use a 2×2 all-ones kernel.
# 3. Apply transposed_overlap_add with stride 1.
# 4. Predict the shape for stride 2 and stride 3.
# 5. Explain why this is not a true inverse.
#
# ES:
# 1. Inspecciona el recorte real 2×2 `small`.
# 2. Usa un kernel 2×2 de unos.
# 3. Aplica transposed_overlap_add con stride 1.
# 4. Predice la forma para stride 2 y stride 3.
# 5. Explica por qué esto no es una inversa verdadera.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

ker = np.ones(
    (2, 2),
    dtype=float,
)

print("Real input / Entrada real:")
print(np.round(small, 3))
print()

for stride in [1, 2, 3]:
    out = transposed_overlap_add(
        small,
        ker,
        stride,
    )

    predicted = (
        (small.shape[0] - 1) * stride + ker.shape[0],
        (small.shape[1] - 1) * stride + ker.shape[1],
    )

    print(
        f"stride/paso={stride} | "
        f"predicted/predicha={predicted} | "
        f"actual/real={out.shape}"
    )

print()
print("EN: the operator expands and combines values; it does not reconstruct a previous unknown image.")
print("ES: el operador expande y combina valores; no reconstruye automáticamente una imagen previa desconocida.")

### Interactive overlap-add explorer / Explorador interactivo de superposición y suma

Change the stride.

The plots show:

1. the real `2×2` input;
2. the transposed-convolution-style output;
3. an overlap-count map showing how many kernel placements contribute to each output position.

> 🇪🇸 Cambia el stride.
>
> Verás la entrada real `2×2`, la salida y un mapa que indica cuántas colocaciones del kernel contribuyen a cada posición.

In [ ]:
stride_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=3,
    step=1,
    description="Stride / Paso:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def overlap_count_map(x_shape, kernel_shape, stride):
    h, w = x_shape
    kh, kw = kernel_shape

    out_h = (h - 1) * stride + kh
    out_w = (w - 1) * stride + kw

    count = np.zeros(
        (out_h, out_w),
        dtype=int,
    )

    for i in range(h):
        for j in range(w):
            r = i * stride
            c = j * stride

            count[
                r:r + kh,
                c:c + kw,
            ] += 1

    return count

def explore_transposed(stride):
    out = transposed_overlap_add(
        small,
        ker,
        stride,
    )

    count = overlap_count_map(
        small.shape,
        ker.shape,
        stride,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(10.5, 3.5),
        constrained_layout=True,
    )

    images = [
        axes[0].imshow(
            small,
            cmap="viridis",
        ),
        axes[1].imshow(
            out,
            cmap="viridis",
        ),
        axes[2].imshow(
            count,
            cmap="Greys",
            vmin=0,
        ),
    ]

    titles = [
        "Real 2×2 input\nEntrada real",
        f"Overlap-add output\nSalida · stride={stride}",
        "Overlap count\nCantidad de superposiciones",
    ]

    for ax, title in zip(axes, titles):
        ax.set_title(title)

        arr = ax.images[0].get_array()

        ax.set_xticks(
            range(arr.shape[1])
        )
        ax.set_yticks(
            range(arr.shape[0])
        )

    for i in range(count.shape[0]):
        for j in range(count.shape[1]):
            axes[2].text(
                j,
                i,
                str(count[i, j]),
                ha="center",
                va="center",
            )

    plt.show()

    predicted = (
        (small.shape[0] - 1) * stride + ker.shape[0],
        (small.shape[1] - 1) * stride + ker.shape[1],
    )

    print("Input / Entrada:", small.shape)
    print("Kernel:", ker.shape)
    print("Stride / Paso:", stride)
    print("Predicted output / Salida predicha:", predicted)
    print("Actual output / Salida real:", out.shape)
    print()
    print("EN: overlap-add explains the shape change; no lost history is recreated.")
    print("ES: la superposición y suma explica el cambio de forma; no se recrea información perdida.")

transpose_output = widgets.interactive_output(
    explore_transposed,
    {"stride": stride_slider},
)

display(widgets.VBox([stride_slider, transpose_output]))

<details>
<summary><strong>Why is transposed convolution not deconvolution? / ¿Por qué la convolución transpuesta no es deconvolución?</strong></summary>

A true inverse would need enough information to determine the earlier unknown signal.

Transposed convolution instead applies the transpose/adjoint-style structure of the forward operator.

It is extremely useful for learned upsampling, decoders, and generative models.

But:

> **larger spatial shape ≠ recovered original**

> 🇪🇸 Una inversa verdadera tendría que recuperar el contenido desconocido anterior.
>
> La convolución transpuesta aplica una operación estructurada de expansión y suma, muy útil en decoders, pero:
>
> **mayor tamaño espacial ≠ original recuperado**

</details>

## Exercise 3 — true deconvolution on a real photograph / Ejercicio 3 — deconvolución verdadera sobre una fotografía real

Now we solve an actual inverse problem.

We begin with a real `256×256` crop.

Then we deliberately:

1. blur it with a known `9×9` point-spread function (PSF);
2. add a small reproducible amount of noise;
3. try to estimate the original using **Richardson–Lucy**.

### Point-spread function (PSF) / Función de dispersión de punto

A PSF describes how an imaging system spreads the information from one ideal point across nearby pixels.

In microscopy, astronomy, and medical imaging, a measured image is often blurred by such an optical response.

### Why measure the interior? / ¿Por qué medir el interior?

Near the image boundary, the algorithm does not know what exists outside the field of view.

That makes border pixels less reliable.

So we measure error after removing a fixed border.

> 🇪🇸 Una PSF describe cómo un sistema de imagen dispersa la información de un punto ideal sobre píxeles vecinos.
>
> Los bordes son menos confiables porque el algoritmo desconoce qué existe fuera del campo de visión; por eso mediremos el error en el interior.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# 1. Build a normalized 9×9 averaging PSF.
# 2. Blur `work` with signal.fftconvolve(..., mode="same").
# 3. Add reproducible Gaussian noise.
# 4. Recover with richardson_lucy.
# 5. Compare interior relative error before and after.
#
# ES:
# 1. Construye una PSF promedio 9×9 normalizada.
# 2. Desenfoca `work` con signal.fftconvolve(..., mode="same").
# 3. Agrega ruido gaussiano reproducible.
# 4. Recupera con richardson_lucy.
# 5. Compara el error relativo interior antes y después.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

psf = np.ones(
    (9, 9),
    dtype=float,
)

psf /= psf.sum()

blurred = signal.fftconvolve(
    work,
    psf,
    mode="same",
)

noise = (
    0.002
    * np.random.default_rng(0).standard_normal(
        work.shape
    )
)

noisy = np.clip(
    blurred + noise,
    0,
    1,
)

border = 20

def interior_relative_error(candidate):
    ref = work[
        border:-border,
        border:-border,
    ]

    cand = candidate[
        border:-border,
        border:-border,
    ]

    return (
        np.linalg.norm(cand - ref)
        / np.linalg.norm(ref)
    )

recovered_20 = richardson_lucy(
    noisy,
    psf,
    num_iter=20,
    clip=False,
)

err_before = interior_relative_error(
    noisy
)

err_after = interior_relative_error(
    recovered_20
)

print(
    "Blurred + noise error / Error desenfoque + ruido:",
    f"{err_before:.4f}",
)

print(
    "Richardson–Lucy 20 iterations / 20 iteraciones:",
    f"{err_after:.4f}",
)

print()

if err_after < err_before:
    print("EN: the interior reconstruction error improved.")
    print("ES: el error de reconstrucción interior mejoró.")
else:
    print("EN: this parameter choice did not improve the interior metric.")
    print("ES: esta elección de parámetros no mejoró la métrica interior.")

### Interactive deconvolution explorer / Explorador interactivo de deconvolución

Move the number of Richardson–Lucy iterations.

Compare:

- original real image;
- blurred + noisy observation;
- recovered image;
- absolute-error map.

The error shown in text is measured on the **interior only**.

> 🇪🇸 Mueve el número de iteraciones de Richardson–Lucy.
>
> Compara la imagen real, la observación desenfocada con ruido, la imagen recuperada y el mapa de error absoluto.
>
> El error numérico se mide únicamente en el **interior**.

In [ ]:
iterations_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=50,
    step=1,
    description="Iterations / Iteraciones:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_deconvolution(iterations):
    recovered = richardson_lucy(
        noisy,
        psf,
        num_iter=iterations,
        clip=False,
    )

    clipped = np.clip(
        recovered,
        0,
        1,
    )

    err_before = interior_relative_error(
        noisy
    )

    err_after = interior_relative_error(
        recovered
    )

    abs_error = np.abs(
        clipped - work
    )

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(14, 3.5),
        constrained_layout=True,
    )

    items = [
        (
            work,
            "Original real\nOriginal real",
            "gray",
            0,
            1,
        ),
        (
            noisy,
            "Blurred + noise\nDesenfoque + ruido",
            "gray",
            0,
            1,
        ),
        (
            clipped,
            f"Recovered · {iterations} iterations\n"
            f"Recuperada · {iterations} iteraciones",
            "gray",
            0,
            1,
        ),
        (
            abs_error,
            "Absolute error\nError absoluto",
            "magma",
            0,
            None,
        ),
    ]

    for ax, (
        image,
        title,
        cmap,
        vmin,
        vmax,
    ) in zip(axes, items):
        im = ax.imshow(
            image,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )

        ax.set_title(
            title,
            fontsize=10,
            pad=10,
        )
        ax.axis("off")

    plt.show()

    print("Iterations / Iteraciones:", iterations)
    print(f"Before / Antes:  {err_before:.4f}")
    print(f"After / Después: {err_after:.4f}")

    change = (
        (err_before - err_after)
        / err_before
    )

    print(
        f"Relative improvement / Mejora relativa: "
        f"{change:.1%}"
    )

    if err_after < err_before:
        print("EN: recovery currently improves the interior metric.")
        print("ES: la recuperación mejora actualmente la métrica interior.")
    else:
        print("EN: at this iteration count, the metric is no longer better than the blurred observation.")
        print("ES: con este número de iteraciones, la métrica ya no es mejor que la observación desenfocada.")

deconv_output = widgets.interactive_output(
    explore_deconvolution,
    {"iterations": iterations_slider},
)

display(widgets.VBox([iterations_slider, deconv_output]))

### How many iterations are useful? / ¿Cuántas iteraciones son útiles?

More iterations do not automatically mean a better reconstruction.

The next explorer computes the interior error over several iteration counts and marks the selected point.

This lets you see whether the iterative inverse problem is still improving or beginning to emphasize noise.

> 🇪🇸 Más iteraciones no significan automáticamente una mejor reconstrucción.
>
> El siguiente explorador calcula el error interior para varios números de iteraciones y marca el punto seleccionado.

In [ ]:
curve_iteration = widgets.IntSlider(
    value=20,
    min=1,
    max=40,
    step=1,
    description="Iteration / Iteración:",
    continuous_update=False,
    style={"description_width": "125px"},
)

iteration_grid = np.arange(
    1,
    41,
)

# Compute once so moving the marker stays responsive.
rl_errors = []

for n_iter in iteration_grid:
    candidate = richardson_lucy(
        noisy,
        psf,
        num_iter=int(n_iter),
        clip=False,
    )

    rl_errors.append(
        interior_relative_error(
            candidate
        )
    )

rl_errors = np.asarray(
    rl_errors
)

best_iter = int(
    iteration_grid[
        np.argmin(rl_errors)
    ]
)

def explore_error_curve(iteration):
    idx = iteration - 1

    fig, ax = plt.subplots(
        figsize=(8.5, 3.8),
        constrained_layout=True,
    )

    ax.plot(
        iteration_grid,
        rl_errors,
        marker="o",
        markersize=3,
        label="RL interior error / error interior RL",
    )

    ax.axhline(
        interior_relative_error(noisy),
        linestyle="--",
        label="blurred + noise / desenfoque + ruido",
    )

    ax.scatter(
        [iteration],
        [rl_errors[idx]],
        s=90,
        zorder=5,
        label="selected / seleccionado",
    )

    ax.set_xlabel("Richardson–Lucy iterations / iteraciones")
    ax.set_ylabel("relative interior error / error relativo interior")
    ax.set_title(
        "Iteration trade-off / Compromiso del número de iteraciones"
    )
    ax.legend()

    plt.show()

    print("Selected iteration / Iteración seleccionada:", iteration)
    print("Selected error / Error seleccionado:", f"{rl_errors[idx]:.4f}")
    print("Best tested iteration / Mejor iteración probada:", best_iter)
    print("Best tested error / Mejor error probado:", f"{rl_errors.min():.4f}")
    print()
    print("EN: iteration count is a model choice; more is not automatically better.")
    print("ES: el número de iteraciones es una decisión del modelo; más no es automáticamente mejor.")

curve_output = widgets.interactive_output(
    explore_error_curve,
    {"iteration": curve_iteration},
)

display(widgets.VBox([curve_iteration, curve_output]))

<details>
<summary><strong>Why is deconvolution difficult? / ¿Por qué es difícil la deconvolución?</strong></summary>

Forward blur is easy:

`original → blur kernel → blurred observation`

The inverse direction is harder:

`blurred + noise → ? → original estimate`

because:

- different originals can produce very similar blurred observations;
- some high-frequency detail may have been strongly suppressed;
- noise is mixed with the useful signal;
- boundaries contain incomplete information.

Richardson–Lucy does not “restore the exact hidden truth.”

It produces an estimate under a mathematical image-formation model.

> 🇪🇸 El desenfoque directo es sencillo, pero la dirección inversa es más difícil porque se pierde o atenúa información y el ruido se mezcla con la señal.
>
> Richardson–Lucy produce una **estimación**, no una recuperación garantizada del original exacto.

</details>

## Quick reasoning challenge / Reto rápido de razonamiento

Choose a situation and identify the operation.

> 🇪🇸 Elige una situación e identifica la operación correcta.

In [ ]:
decision_choice = widgets.Dropdown(
    options=[
        (
            "Slide Sobel weights without flipping / "
            "Deslizar Sobel sin invertir",
            "correlation",
        ),
        (
            "Flip kernel, then slide / "
            "Invertir kernel y deslizar",
            "convolution",
        ),
        (
            "Upsample with learned overlap-add / "
            "Aumentar tamaño con superposición aprendida",
            "transpose",
        ),
        (
            "Estimate sharp image from known blur / "
            "Estimar imagen nítida desde desenfoque conocido",
            "deconvolution",
        ),
    ],
    value="correlation",
    description="Situation / Situación:",
    style={"description_width": "130px"},
)

def explain_decision(choice):
    answers = {
        "correlation": (
            "Cross-correlation",
            "Correlación cruzada",
        ),
        "convolution": (
            "True convolution",
            "Convolución verdadera",
        ),
        "transpose": (
            "Transposed convolution",
            "Convolución transpuesta",
        ),
        "deconvolution": (
            "Deconvolution / inverse problem",
            "Deconvolución / problema inverso",
        ),
    }

    en, es = answers[choice]

    print("EN:", en)
    print("ES:", es)

decision_output = widgets.interactive_output(
    explain_decision,
    {"choice": decision_choice},
)

display(widgets.VBox([decision_choice, decision_output]))

## What just happened / Qué acaba de pasar

You followed one local operator through four different ideas.

### 1. Correlation vs convolution / Correlación vs convolución

Correlation slides the kernel as written.

True convolution flips the kernel first.

For the Sobel example:

`correlation(image, K) = convolution(image, flip(K))`

### 2. Toeplitz view / Vista Toeplitz

The convolution of 32 real pixel measurements became:

`C @ x`

The algebra was ordinary matrix multiplication.

The special part was the repeated shifted structure inside `C`.

### 3. Transposed convolution / Convolución transpuesta

Overlap-add changed the spatial shape.

But:

> **shape expansion is not information recovery**

### 4. True deconvolution / Deconvolución verdadera

Richardson–Lucy used:

- a known PSF;
- a noisy blurred observation;
- repeated iterative updates;

to estimate a sharper image.

The result had to be evaluated carefully because inverse problems are sensitive to noise and boundaries.

### The sentence to remember / La frase para recordar

> **Convolution applies a structured forward operator; transposed convolution applies its shape-changing partner; deconvolution tries to solve the inverse problem.**

> 🇪🇸
>
> **La convolución aplica un operador directo estructurado; la convolución transpuesta aplica su compañero que cambia la forma; la deconvolución intenta resolver el problema inverso.**

### Final self-check / Autoevaluación final

If an operation changes:

`16×16 → 32×32`

does that prove it reconstructed a lost `32×32` original?

**No. Output size alone says nothing about whether lost information was recovered.**

> 🇪🇸 Si una operación cambia `16×16 → 32×32`, ¿eso demuestra que reconstruyó un original perdido de `32×32`?
>
> **No. El tamaño de salida por sí solo no demuestra que se haya recuperado información perdida.**

---

## Done with this section / Fin de esta sección

Next / Siguiente: **10 · Tucker decomposition on real data / Descomposición Tucker con datos reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)